Finished Logreg!!

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA 

# Load the data
X_train = pd.read_csv('Datasets/cleaned_X_train2.csv')
y_train = pd.read_csv('Datasets/y_train.csv')
X_test = pd.read_csv('Datasets/cleaned_X_test2.csv')
y_test = pd.read_csv('Datasets/y_test.csv')

# Convert the labels to a single column of labels
y_train = y_train.idxmax(axis=1)
y_test = y_test.idxmax(axis=1)

# Encode the labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Create the pipeline
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),  
    ('scaler', StandardScaler()), 
    ('pca', PCA(n_components=0.76)),  # Apply PCA with 95% explained variance
    ('oversampler', RandomOverSampler(random_state=42)), 
    ('logreg', LogisticRegressionCV(cv=3, max_iter=1000, multi_class='ovr', solver='liblinear', n_jobs=-1, class_weight='balanced'))  
])  

# Define the hyperparameter search space
param_grid = {
    'logreg__Cs': [0.1, 1, 10],  
    'logreg__max_iter': [1000], 
    'logreg__solver': ['liblinear'],
}

# Perform hyperparameter search with GridSearchCV
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=2)

# Train the model
grid_search.fit(X_train, y_train_encoded)

# Display the best hyperparameters found
print("Best Parameters:", grid_search.best_params_)

# Get the best model from the grid search
best_model = grid_search.best_estimator_

# Predict on the training set and display results
y_train_pred = best_model.predict(X_train)
print(f"Train Accuracy: {accuracy_score(y_train_encoded, y_train_pred):.3f}")
print("Training Classification Report:")
print(classification_report(y_train_encoded, y_train_pred))

# Predict on the test set and display results
y_test_pred = best_model.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test_encoded, y_test_pred):.3f}")
print(classification_report(y_test_encoded, y_test_pred))




The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Fitting 3 folds for each of 3 candidates, totalling 9 fits


/home/iyoas/.local/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:425: FitFailedWarning: 
3 fits failed out of a total of 9.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3 fits failed with the following error:
Traceback (most recent call last):
  File "/home/iyoas/.local/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 729, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/iyoas/.local/lib/python3.10/site-packages/sklearn/base.py", line 1152, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/home/iyoas/miniconda3/lib/python3.10/site-packages/imblearn/pipeline.py", line 526, in fit
    self._final_estimator.fit(Xt, yt, **last_step_params[

Best Parameters: {'logreg__Cs': 1, 'logreg__max_iter': 1000, 'logreg__solver': 'liblinear'}
Train Accuracy: 0.394
Training Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.48      0.47        48
           1       0.20      0.76      0.31        37
           2       0.50      0.12      0.20        41
           3       0.60      0.38      0.46        48
           4       0.33      0.23      0.27        39
           5       0.89      0.66      0.76        50
           6       1.00      0.08      0.15        62
           7       0.31      0.42      0.36        24
           8       0.39      0.56      0.46        25
           9       0.39      0.47      0.42        30

    accuracy                           0.39       404
   macro avg       0.51      0.41      0.39       404
weighted avg       0.56      0.39      0.38       404

Test Accuracy: 0.218
              precision    recall  f1-score   support

           0       

/home/iyoas/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/iyoas/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/iyoas/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


With newer dataset but results are worse

In [19]:
import pandas as pd
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.compose import ColumnTransformer

# Load data
X_train = pd.read_csv('Datasets/alternative_X_train.csv')
y_train = pd.read_csv('Datasets/y_train.csv')
X_test = pd.read_csv('Datasets/alternative_X_test.csv')
y_test = pd.read_csv('Datasets/y_test.csv')

# Convert labels to single-column format
y_train = y_train.idxmax(axis=1)
y_test = y_test.idxmax(axis=1)

# Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

# Identify categorical and numerical columns
categorical_cols = X_train.select_dtypes(include=['object']).columns
numerical_cols = X_train.select_dtypes(exclude=['object']).columns

# Define preprocessing for categorical and numerical features separately
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]), numerical_cols),
    
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]), categorical_cols)
])

# Define the pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('oversampler', RandomOverSampler(random_state=42)),  # Move oversampling before PCA
    ('pca', PCA(n_components=0.76)),  # Apply PCA after balancing
    ('logreg', LogisticRegressionCV(cv=3, max_iter=1000, multi_class='ovr', solver='liblinear', n_jobs=-1, class_weight='balanced'))
])

# Define hyperparameter grid
param_grid = {
    'logreg__Cs': [1, 10],  # Remove invalid values for LogisticRegressionCV
    'logreg__max_iter': [1000],
    'logreg__solver': ['liblinear']
}

# Perform Grid Search
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train_encoded)

# Best hyperparameters
print("Best Parameters:", grid_search.best_params_)

# Get the best model
best_model = grid_search.best_estimator_

# Predictions on training data
y_train_pred = best_model.predict(X_train)
print(f"Train Accuracy: {accuracy_score(y_train_encoded, y_train_pred):.3f}")
print("Train Classification Report:")
print(classification_report(y_train_encoded, y_train_pred))

# Predictions on test data
y_test_pred = best_model.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test_encoded, y_test_pred):.3f}")
print("Test Classification Report:")
print(classification_report(y_test_encoded, y_test_pred))

# Confusion Matrix
print("Confusion Matrix (Test):")
print(confusion_matrix(y_test_encoded, y_test_pred))


Fitting 3 folds for each of 2 candidates, totalling 6 fits


/home/marilene/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1917: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegressionCV(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/marilene/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1917: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegressionCV(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/marilene/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1917: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegressionCV(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
/home/marilene/.local/lib/python3.10/site-packages/sklearn/linear_model/_logist

[CV] END logreg__Cs=1, logreg__max_iter=1000, logreg__solver=liblinear; total time=   0.6s
[CV] END logreg__Cs=1, logreg__max_iter=1000, logreg__solver=liblinear; total time=   0.8s
[CV] END logreg__Cs=1, logreg__max_iter=1000, logreg__solver=liblinear; total time=   0.8s
[CV] END logreg__Cs=10, logreg__max_iter=1000, logreg__solver=liblinear; total time=   2.8s
[CV] END logreg__Cs=10, logreg__max_iter=1000, logreg__solver=liblinear; total time=   2.8s
[CV] END logreg__Cs=10, logreg__max_iter=1000, logreg__solver=liblinear; total time=   2.9s


/home/marilene/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1917: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegressionCV(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


Best Parameters: {'logreg__Cs': 1, 'logreg__max_iter': 1000, 'logreg__solver': 'liblinear'}
Train Accuracy: 0.460
Train Classification Report:
              precision    recall  f1-score   support

           0       0.40      0.54      0.46        48
           1       0.42      0.51      0.46        37
           2       0.41      0.41      0.41        41
           3       0.44      0.29      0.35        48
           4       0.49      0.51      0.50        39
           5       0.53      0.62      0.57        50
           6       0.58      0.24      0.34        62
           7       0.34      0.46      0.39        24
           8       0.42      0.52      0.46        25
           9       0.61      0.67      0.63        30

    accuracy                           0.46       404
   macro avg       0.46      0.48      0.46       404
weighted avg       0.47      0.46      0.45       404

Test Accuracy: 0.089
Test Classification Report:
              precision    recall  f1-score   sup

/home/marilene/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/marilene/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/marilene/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
